#Baseline Evaluation

In [ ]:
#This clones the github repo codebase into Google Colab
!git clone https://github.com/api-sage/recsys2025.git /kaggle/working/recsys2025

In [ ]:
import sys, os, importlib, traceback

os.chdir('/kaggle/working/recsys2025')
repo = '/kaggle/working/recsys2025'
if repo not in sys.path:
    sys.path.insert(0, repo)

# clear any cached/failed imports
for m in [m for m in sys.modules if m.startswith('data_utils')]:
    del sys.modules[m]
importlib.invalidate_caches()

# which data_utils would Python pick?
spec = importlib.util.find_spec('data_utils')
print("data_utils resolves to:", spec.origin if spec else None)

# try the import and show the full traceback
try:
    from data_utils.data_dir import DataDir
    print("import OK")
except Exception:
    traceback.print_exc()

In [ ]:
import os, sys, importlib

os.makedirs("/kaggle/working/baseline_scores", exist_ok=True)
repo = '/kaggle/working/recsys2025'
os.chdir(repo)
if repo not in sys.path:
    sys.path.insert(0, repo)
importlib.invalidate_caches()

from data_utils.data_dir import DataDir

In [ ]:
#Installation of specified libraries and packages the codebase application demands
!pip install -r requirements.txt
#%pip install -r "C:\Users\afola\Documents\recsys_group_project\recsys2025\requirements.txt"

In [ ]:
!ls /kaggle/input/datasets/pauloafolabi/challenge-dataset

In [ ]:
from pathlib import Path

drive_dataset_path = Path("/kaggle/input/datasets/pauloafolabi/challenge-dataset/challenge_dataset")
dataset = DataDir(drive_dataset_path)

print(f"Input directory:: {dataset.input_dir}")
print(f"Target directory:: {dataset.target_dir}")
print(f"Property file:: {dataset.properties_file}")

In [ ]:
for p in sorted(drive_dataset_path.rglob("*")):
    if p.is_file():
        print(p.relative_to(drive_dataset_path), f"{p.stat().st_size/1e6:.1f} MB")

In [ ]:
import shutil
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from pathlib import Path

src = drive_dataset_path
dst = Path("/kaggle/working/challenge_dataset_30")
FRAC, SEED = 0.3, 42

# choose 10% of the relevant clients
relevant = np.load(src / "input" / "relevant_clients.npy")
rng = np.random.default_rng(SEED)
keep = np.sort(rng.choice(relevant, size=int(len(relevant) * FRAC), replace=False))

def filter_parquet(f, out):
    pf = pq.ParquetFile(f)
    schema = pf.schema_arrow
    keep_arr = pa.array(keep).cast(schema.field("client_id").type)
    with pq.ParquetWriter(out, schema) as writer:
        for batch in pf.iter_batches(batch_size=2_000_000):
            tbl = pa.Table.from_batches([batch])
            mask = pc.is_in(tbl["client_id"], value_set=keep_arr)
            writer.write_table(tbl.filter(mask))

for f in sorted(src.rglob("*")):
    if not f.is_file():
        continue
    out = dst / f.relative_to(src)
    out.parent.mkdir(parents=True, exist_ok=True)

    if f.suffix == ".parquet" and "client_id" in pq.ParquetFile(f).schema_arrow.names:
        filter_parquet(f, out)          # events + train/validation targets
    elif f.name in ("relevant_clients.npy", "active_clients.npy"):
        arr = np.load(f)
        np.save(out, arr[np.isin(arr, keep)])
    else:
        shutil.copy(f, out)             # product_properties, propensity/popularity files
    print("done:", f.relative_to(src))

dataset = DataDir(dst)
print(dataset.input_dir, dataset.target_dir, dataset.properties_file)

In [ ]:
# Run the embedding creation script
#!python -m baseline.aggregated_features_baseline.create_embeddings --data-dir "{drive_dataset_path}" --embeddings-dir ./outputs
!python -m baseline.aggregated_features_baseline.create_embeddings \
    --data-dir "/kaggle/working/challenge_dataset_30" \
    --embeddings-dir /kaggle/working/outputs

In [ ]:
#!python -m validator.validate --embeddings-path ./outputs --data-dir ./data
!python -m validator.run \
    --data-dir "/kaggle/working/challenge_dataset_10" \
    --embeddings-dir /kaggle/working/outputs

In [ ]:
!pip install "pytorch-lightning==2.4.0" neptune

In [ ]:
!{sys.executable} -m training_pipeline.train \
    --data-dir "/kaggle/working/challenge_dataset_30" \
    --embeddings-dir /kaggle/working/outputs \
    --tasks churn propensity_category propensity_sku \
    --log-name baseline_eval \
    --accelerator gpu \
    --devices 0 \
    --num-workers 2 \
    --score-dir /kaggle/working/baseline_scores

In [ ]:
import os
for file in os.listdir("/kaggle/working/baseline_scores"):
    print(f"=== {file} ===")
    with open(f"/kaggle/working/baseline_scores/{file}") as f:
        print(f.read())